# A11 · Dates and time series: 24 years of rain at Melamchi

> **Time:** about 2 hours  
> **You need:** Lessons A1 to A10, especially pandas tables (A9) and plotting with Matplotlib (A10).  
> **By the end you can:**
> 1. turn text into real dates, take dates apart with `.dt`, and select days, months and years from a table indexed by date,
> 2. summarise daily values into months and years with `resample` and `groupby`, and compute rolling and running totals,
> 3. compare two places, find extremes and anomalies, and explain what model rainfall data can and cannot show.

Heavy rain is a common trigger of landslides in Nepal, and rain is recorded day by day. A table with one row per day is a **time series**. In this lesson you analyse 24 years of daily rainfall, from 2001 to 2024, at Melamchi (Sindhupalchok) and at Kathmandu.

**Where the numbers come from.** These are **real data, but not rain-gauge measurements.** They come from **ERA5**, a **reanalysis**: a weather model that is corrected with millions of observations to estimate the weather everywhere on Earth. Each daily value is an average over a square about 25–30 km wide, not the rain at one spot. Such averages are very useful, but they smooth out short, local downpours. Step 8 shows what that means for the June 2021 Melamchi flood.

*Weather data by Open-Meteo.com (CC BY 4.0), from the ERA5 reanalysis of the Copernicus Climate Change Service.* The file and its source are described in `data/README.md`.

**Counting lines in this lesson:** as in A10, `fig, ax = plt.subplots()` and `plt.show()` are not counted in the Q1/Q2/Q3 line counts.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 11 of 39 · [Course home](../README.md)  
← Previous: [A10 · Plotting with Matplotlib: charts of rain and mountains](A10_Plotting_with_Matplotlib.ipynb) · Next: [A12 · Scripts, modules and good habits: tools for real projects](A12_Scripts_modules_and_good_habits.ipynb) →

## 1. Dates are not text

A CSV file stores everything as characters. When pandas reads `2021-06-15`, it does not know that this is a date. It keeps it as **text**, a string (A2).

Text dates look right, but they behave wrong. You cannot subtract one from another, and pandas cannot tell which month they belong to.

`pd.to_datetime(text)` converts text into real dates. pandas calls one date a **Timestamp**, and a column of them a **datetime** column. `pd.read_csv(path, parse_dates=["date"])` does the conversion while reading.

Subtracting two Timestamps gives a **Timedelta**: a length of time, such as 30 days.

**Predict:** the file has a `date` column. What type will its first value have when you read the file without `parse_dates`?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)

raw = pd.read_csv(DATA / "rain_daily_melamchi_kathmandu.csv")
print(raw.head(3))
print(raw.shape)
print(type(raw["date"][0]))

The table has 17,532 rows: one per day and place. The first value of `date` is a `str`. It prints like a date, but to Python it is a row of characters, like `"Everest"`. Watch what happens when you ask for the length of the record:

In [ ]:
raw["date"].max() - raw["date"].min()

`TypeError: unsupported operand type(s) for -: 'str' and 'str'`. Python cannot subtract text from text (A2, A6). Read the file again, this time with `parse_dates`:

In [ ]:
weather = pd.read_csv(DATA / "rain_daily_melamchi_kathmandu.csv", parse_dates=["date"])
print(type(weather["date"][0]))

first_day = weather["date"].min()
last_day = weather["date"].max()
span = last_day - first_day
print(first_day, "to", last_day)
print(span, "=", round(span.days / 365.25, 1), "years")

Now each date is a `Timestamp`, and subtraction works. The record runs from 1 January 2001 to 31 December 2024. The difference is a `Timedelta` of 8,765 days; `span.days` gives the number of days as a plain number. Midnight, `00:00:00`, is shown because a Timestamp can also hold a time of day.

**Other date formats.** This file writes dates as year-month-day (`2021-06-15`), the international standard called **ISO 8601**. Field notes often write day first, `15/06/2021`. Tell pandas the pattern with `format=`: `%d` is the day, `%m` the month and `%Y` the four-digit year.

### Practice

**Q1 · one line:** How many days passed between the Gorkha earthquake (25 April 2015, facts sheet) and the last day of the record? Use `last_day` from the example.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(last_day - pd.Timestamp("2015-04-25"))
```

`pd.Timestamp("2015-04-25")` makes a single date from text. The difference is 3,538 days.
</details>

**Q2 · two lines:** A field notebook writes the earthquake date day first: `"25/04/2015"`. Convert it with `pd.to_datetime` and the right `format=`, then print whether it equals `pd.Timestamp("2015-04-25")`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
quake = pd.to_datetime("25/04/2015", format="%d/%m/%Y")
print(quake == pd.Timestamp("2015-04-25"))
```

It prints `True`. The format string tells pandas where to find each part, so day-first and year-first text become the same date.
</details>

**Q3 · three lines:** Three field visits are written as `["02/07/2021", "15/06/2021", "28/05/2021"]` (made-up dates for practice). Print the list sorted as text, then convert it with `pd.to_datetime(..., format="%d/%m/%Y")` and print it sorted as dates. Which order is right?

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
visits = ["02/07/2021", "15/06/2021", "28/05/2021"]
print(sorted(visits))
print(pd.to_datetime(visits, format="%d/%m/%Y").sort_values())
```

Sorted as text, "02/07" comes first because the character `0` comes before `1` and `2`. That puts 2 July before 28 May. Sorted as dates, the order is 28 May, 15 June, 2 July. Only real dates sort in time order, whatever format they were written in.
</details>

## 2. Taking a date apart: the `.dt` tools

A datetime column has extra tools behind the name `.dt`. Such a group of tools behind a dot is called an **accessor**:

- `.dt.year`, `.dt.month` and `.dt.day` give the parts of each date as whole numbers,
- `.dt.day_of_year` gives the day of the year, from 1 (1 January) to 366,
- `.dt.day_name()` gives the weekday as text.

Each one returns a Series with one value per row. Store it as a new column (A9), and you can filter and group by year or month.

`weather[weather["place"] == "Melamchi"]` keeps the Melamchi rows. Add `.copy()` to make it a separate table, so that changing it later never touches `weather`.

**Predict:** how many rows will each year have at Melamchi? Is it the same number every year?

In [ ]:
weather["year"] = weather["date"].dt.year
weather["month"] = weather["date"].dt.month
weather["doy"] = weather["date"].dt.day_of_year      # day of year, 1 to 366
print(weather.head(3))
print(weather["date"].dt.day_name().head(3))

In [ ]:
mel = weather[weather["place"] == "Melamchi"].copy()
days_per_year = mel.groupby("year")["rain_mm"].count()
print(days_per_year.head(5))
print(len(mel), "days at Melamchi")

Most years have 365 rows, but 2004 has 366: it is a **leap year**, with 29 February. The record holds six leap years (2004, 2008, 2012, 2016, 2020 and 2024), so Melamchi has 24 × 365 + 6 = 8,766 days. 1 January 2001 was a Monday.

Counting rows per year is a quick check that no days are missing. A gap in a time series can hide the very storm you are looking for.

### Practice

**Q1 · one line:** On which weekday did the Gorkha earthquake (25 April 2015) happen? A single Timestamp has the same tools, without `.dt`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
pd.Timestamp("2015-04-25").day_name()
```

It was a Saturday. On a single Timestamp you write `.day_name()` or `.year` directly; `.dt` is only needed for a whole column.
</details>

**Q2 · two lines:** Use the `month` column of `mel` to print the mean daily rainfall at Melamchi in January and in July.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(mel.loc[mel["month"] == 1, "rain_mm"].mean())
print(mel.loc[mel["month"] == 7, "rain_mm"].mean())
```

An average January day gets about 1.2 mm, an average July day about 25 mm, more than twenty times as much. The filter uses `.loc[mask, column]` from A9.
</details>

**Q3 · three lines:** Use the `year` column of `mel` to compare two years: print the total rainfall at Melamchi in 2015 (the year of the Gorkha earthquake) and in 2021 (the year of the Melamchi flood), rounded to 1 decimal place.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
total_2015_mm = mel.loc[mel["year"] == 2015, "rain_mm"].sum()
total_2021_mm = mel.loc[mel["year"] == 2021, "rain_mm"].sum()
print(round(total_2015_mm, 1), round(total_2021_mm, 1))
```

2015 had 2,095.9 mm and 2021 had 3,697.1 mm. Step 4 shows a shorter way to get the total of every year at once, and that these are the driest and the wettest years of the record.
</details>

## 3. A date index: selecting days, months and ranges

`mel.set_index("date")` makes the dates the row labels. pandas calls this a **DatetimeIndex**, and it makes `.loc` (A9) understand dates written as text:

- `.loc["2021-06-15"]`: one day,
- `.loc["2021-06"]`: a whole month,
- `.loc["2021"]`: a whole year,
- `.loc["2021-06-12":"2021-06-18"]`: a range of days, with **both ends included**, as with labels in A9.

`.idxmax()` gives the **label** of the largest value, here its date; `.idxmin()` the label of the smallest. `.date()` shows a Timestamp without the time.

If the results of these cells ever look strange, use **Kernel → Restart Kernel and Run All Cells**: an old variable from an earlier run may still be in memory.

**Predict:** how many rows will `rain_mm.loc["2021-06"]` have?

In [ ]:
mel_daily = mel.set_index("date")
rain_mm = mel_daily["rain_mm"]          # daily rainfall at Melamchi, a Series with a DatetimeIndex

print(rain_mm.loc["2021-06-15"])
june_2021 = rain_mm.loc["2021-06"]
print(len(june_2021), "days, total", round(june_2021.sum(), 1), "mm")
print(rain_mm.loc["2021-06-12":"2021-06-18"])

In [ ]:
rain_2021 = rain_mm.loc["2021"]

fig, ax = plt.subplots(figsize=(10, 3))      # fig: the whole image; ax: the plotting area inside it
ax.bar(rain_2021.index, rain_2021, width=1)  # width=1: each bar is one day wide
ax.set_title("Daily rainfall at Melamchi, 2021 (ERA5 reanalysis)")
ax.set_xlabel("Date")
ax.set_ylabel("Rainfall (mm per day)")
plt.show()

June 2021 has 30 rows, one per day, with 519.8 mm in total. On 15 June the reanalysis shows 58.8 mm.

In the chart, Matplotlib reads the dates on the x-axis and writes month labels for you. Bars suit daily rainfall, because each value is an amount that starts at zero (A10). The year has a clear shape: dry months from November to March, then the monsoon from June to September.

### Practice

**Q1 · one line:** Find the wettest day of the whole record at Melamchi: print the largest value of `rain_mm` and its date.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(rain_mm.max(), rain_mm.idxmax().date())
```

The wettest day is 15 August 2014, with 211.8 mm. `idxmax` returns the date label, not the position.
</details>

**Q2 · two lines:** Print the rainfall at Melamchi on the day of the Gorkha earthquake (2015-04-25) and the total for April 2015.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(rain_mm.loc["2015-04-25"])
print(round(rain_mm.loc["2015-04"].sum(), 1))
```

The reanalysis shows 11.7 mm on the day and 143.8 mm for the whole of April 2015. The same `.loc` gives one value for a full date and a whole month for a partial one.
</details>

**Q3 · three lines:** Plot the daily mean air temperature (`temp_mean_c`) at Melamchi in 2021 as a line, with a title and axis labels. Then print the dates of the warmest and the coolest day.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
temp_2021_c = mel_daily["temp_mean_c"].loc["2021"]
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(temp_2021_c.index, temp_2021_c)
ax.set_title("Daily mean air temperature at Melamchi, 2021 (ERA5 reanalysis)")
ax.set_ylabel("Temperature (°C)")
plt.show()
print(temp_2021_c.idxmax().date(), temp_2021_c.idxmin().date())
```

Temperature climbs from January to a broad summer high from June to September, then falls again. The warmest day was 2021-07-23 and the coolest 2021-01-30. A line suits temperature, which is not an amount that adds up like rain.
</details>

## 4. From days to months and years: `resample`

`resample` groups rows by **time period** and summarises each group, like `groupby` (A9) for time. You give it a code for the period, then a summary such as `.sum()` or `.mean()`:

| Code | Period | Each result is labelled with |
| --- | --- | --- |
| `"ME"` | month | the month's **e**nd, such as 2021-06-30 |
| `"MS"` | month | the month's **s**tart, such as 2021-06-01 |
| `"YE"` | year | the year's end, 31 December |

Older pandas versions and older tutorials use `"M"` and `"Y"`. Newer pandas warns about them or refuses them, so use `"ME"` and `"YE"`.

Choose the summary that fits the quantity. Rain adds up, so a monthly **total** is a `.sum()`. Temperature does not add up, so a monthly value is a `.mean()`.

An **anomaly** is the difference between a value and the long-term mean. A positive anomaly means a wetter year than usual, a negative one a drier year.

In [ ]:
monthly_mm = rain_mm.resample("ME").sum()
print(monthly_mm.head(3))
print(len(monthly_mm), "months")

annual_mm = rain_mm.resample("YE").sum()
print(annual_mm.tail(4).round(1))
print("mean:", round(annual_mm.mean(), 1), "mm per year")
print("driest:", annual_mm.idxmin().year, "  wettest:", annual_mm.idxmax().year)

In [ ]:
anomaly_mm = annual_mm - annual_mm.mean()        # each year minus the 24-year mean
colours = np.where(anomaly_mm > 0, "tab:blue", "tab:orange")   # A8

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(anomaly_mm.index.year, anomaly_mm, color=colours)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Annual rainfall at Melamchi: difference from the 2001–2024 mean (ERA5)")
ax.set_xlabel("Year")
ax.set_ylabel("Anomaly (mm)")
plt.show()

24 years give 288 monthly totals. The mean annual total is 2,656.5 mm. The driest year is 2015 (2,095.9 mm) and the wettest is 2021 (3,697.1 mm), followed by 2020.

`anomaly_mm.index.year` turns the year-end dates into plain years for the x-axis. Blue bars are wetter years than the mean, orange bars drier ones. 2021, the year of the flood, stands out as the wettest year of the whole record.

### Practice

**Q1 · one line:** Print the annual **mean** air temperature at Melamchi for the first five years, rounded to 2 decimal places. Use `mel_daily["temp_mean_c"]`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(mel_daily["temp_mean_c"].resample("YE").mean().round(2).head())
```

Temperature is averaged, not summed. The five annual means lie between about 20.8 and 21.2 °C.
</details>

**Q2 · two lines:** Which single month in the record was the wettest? Print its total and its date label from `monthly_mm`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round(monthly_mm.max(), 1))
print(monthly_mm.idxmax().date())
```

July 2007 had 1,200.5 mm. Its label is 2007-07-31 because `"ME"` labels each month by its last day.
</details>

**Q3 · three lines:** Express the anomalies in **percent** of the mean: anomaly ÷ mean × 100. Print the values for 2015 and 2021, rounded to 1 decimal place. Use `.loc["2015"]` on the annual Series.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
anomaly_pct = anomaly_mm / annual_mm.mean() * 100
print(anomaly_pct.loc["2015"].round(1))
print(anomaly_pct.loc["2021"].round(1))
```

2015 was 21.1% drier than the mean and 2021 was 39.2% wetter. Percent anomalies let you compare a wet place with a dry one on the same scale.
</details>

## 5. The average year: grouping by month

`resample` keeps time in order: January 2001, February 2001, and so on, 288 months. To see what a **typical** July looks like, put all 24 Julys together.

That is a `groupby` (A9). It accepts any list of keys with one key per row, such as `monthly_mm.index.month`, the month number of each label. The mean of each group gives 12 values: the average year, called a **climatology**.

Average the monthly **totals**, not the daily values: the result is then in mm per month.

**Predict:** which month is the wettest in the average year, and roughly what share of the year's rain falls from June to September?

In [ ]:
clim_mm = monthly_mm.groupby(monthly_mm.index.month).mean()
print(clim_mm.round(1))
print("sum of the average year:", round(clim_mm.sum(), 1), "mm")

In [ ]:
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(month_names, clim_mm, color="tab:blue")
ax.set_title("Average monthly rainfall at Melamchi, 2001–2024 (ERA5)")
ax.set_xlabel("Month")
ax.set_ylabel("Rainfall (mm per month)")
plt.show()

July is the wettest month, with 774.3 mm in an average year, and November the driest, with 13.6 mm. The 12 monthly means add up to the same 2,656.5 mm as the mean annual total, a useful check.

This chart is the same kind as A10's practice station charts, but built from 8,766 real daily values instead of 12 typed numbers.

### Practice

**Q1 · one line:** What share of the average year's rain falls from June to September? Print it in percent, rounded to 1 decimal place. `clim_mm` has the month numbers 1 to 12 as labels.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round(clim_mm.loc[6:9].sum() / clim_mm.sum() * 100, 1))
```

78.8%: four monsoon months bring almost four fifths of the year's rain. `.loc[6:9]` selects the labels 6 to 9, both included.
</details>

**Q2 · two lines:** Make the climatology of daily mean **temperature**: group `mel_daily["temp_mean_c"]` by month with `mel_daily.index.month`, take the mean, and print it rounded to 1 decimal place. Which months are the warmest and the coolest?

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
clim_temp_c = mel_daily["temp_mean_c"].groupby(mel_daily.index.month).mean()
print(clim_temp_c.round(1))
```

July and August are the warmest months, both 26.4 °C on average, and January the coolest, 13.2 °C. At Melamchi the warm season and the wet season are the same months.
</details>

**Q3 · three lines:** How much does July vary from year to year? Keep only the July totals of `monthly_mm` (where `monthly_mm.index.month == 7`), then print the driest and the wettest July with their years.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
july_mm = monthly_mm[monthly_mm.index.month == 7]
print(july_mm.idxmin().year, round(july_mm.min(), 1))
print(july_mm.idxmax().year, round(july_mm.max(), 1))
```

The driest July (2001, 398.1 mm) had only about a third of the rain of the wettest (2007, 1,200.5 mm). An average of 774.3 mm hides this spread: always look at the years behind a climatology.
</details>

## 6. Rolling windows and running totals

A landslide often follows several days of rain, not one wet day alone. Rain of the previous days fills the soil with water. Geologists call it **antecedent rainfall**.

Researchers use it to set **rainfall thresholds**: amounts of rain over a number of days above which landslides become likely in a region. Real thresholds come from studies of local landslide records; this lesson does not use any.

`rain_mm.rolling(3).sum()` computes a **rolling window** total: for each day, the sum of that day and the two days before. The first two days get `NaN`, short for "not a number": pandas' mark for a **missing value**. They have no full window.

`.cumsum()` gives a **running total**, the cumulative sum, like the `+=` loop of A3. To restart it every 1 January, group by year first: `rain_mm.groupby(rain_mm.index.year).cumsum()`.

**Predict:** on 15 June 2021, will the 3-day total be much larger than that day's 58.8 mm?

In [ ]:
rain_3d_mm = rain_mm.rolling(3).sum()
rain_7d_mm = rain_mm.rolling(7).sum()

windows = pd.DataFrame({"1 day": rain_mm, "3 days": rain_3d_mm, "7 days": rain_7d_mm})
print(windows.loc["2021-06-12":"2021-06-18"].round(1))
print("largest 3-day total:", round(rain_3d_mm.max(), 1), "mm, ending", rain_3d_mm.idxmax().date())

In [ ]:
running_mm = rain_mm.groupby(rain_mm.index.year).cumsum()       # restarts every 1 January
average_run_mm = running_mm.groupby(running_mm.index.day_of_year).mean()
average_run_mm = average_run_mm.loc[1:365]      # day 366 exists only in the 6 leap years
run_2021 = running_mm.loc["2021"]
print("30 June 2021:", round(run_2021.loc["2021-06-30"], 1), "mm; average year:", round(average_run_mm.loc[181], 1), "mm")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(average_run_mm.index, average_run_mm, color="grey", label="Average year, 2001–2024")
ax.plot(run_2021.index.day_of_year, run_2021, color="tab:blue", label="2021")
ax.set_title("Rainfall since 1 January at Melamchi (ERA5)")
ax.set_xlabel("Day of the year")
ax.set_ylabel("Cumulative rainfall (mm)")
ax.legend()
plt.show()

`pd.DataFrame({...})` built a table from three Series with the same dates; each name became a column.

The 3-day total on 15 June 2021 was 98.1 mm, much more than the day's own 58.8 mm, because the two days before were wet too. The largest 3-day total of the whole record, 445.3 mm, ended on 13 August 2017.

The running totals show that 2021 began drier than the average year. From late May it climbed above the average line and stayed there: by 30 June (day 181) it had 1,023.7 mm, against 832.2 mm in the average year. The steep part of each curve is the monsoon.

The average leaves out day 366: only the six leap years have it, so its average would mix a different set of years and jump at the end of the line. Small checks like this keep a chart honest.

### Practice

**Q1 · one line:** Print the largest 7-day total in the record and the day it ends on.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round(rain_7d_mm.max(), 1), rain_7d_mm.idxmax().date())
```

The wettest week ends on 30 July 2007, with 618.4 mm: almost a quarter of an average year in seven days.
</details>

**Q2 · two lines:** Print the largest 3-day total within June 2021 and the day it ends on.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
june_3d_mm = rain_3d_mm.loc["2021-06"]
print(round(june_3d_mm.max(), 1), june_3d_mm.idxmax().date())
```

The wettest 3 days of June 2021 end on 16 June, with 100.4 mm. That is less than a quarter of the record's largest 3-day total of 445.3 mm.
</details>

**Q3 · three lines:** On which day did the running total of 2015 first reach 1,000 mm, and on which day did 2021's? Filter each year's running total to the days with at least 1,000 mm, then take the first date with `.index[0]`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
run_2015 = running_mm.loc["2015"]
print(run_2015[run_2015 >= 1000].index[0].date())
print(run_2021[run_2021 >= 1000].index[0].date())
```

2015 reached 1,000 mm on 2015-07-13, while 2021 got there on 2021-06-30, about two weeks earlier. Filtering with a mask keeps the dates in order, so the first label is the first day the total was reached.
</details>

## 7. Two places side by side: long and wide tables

`weather` is a **long** table: one row per place and day, with the place name in a column. Long tables are good for storing data and for `groupby`.

To compare places day by day, a **wide** table is easier: one row per day and one column per place. `weather.pivot(index="date", columns="place", values="rain_mm")` builds it. Every pair of index and column must appear only once.

`pivot_table` works the same way, but it also **summarises**: with `aggfunc="sum"` it adds up all the rows that share an index and a column, for example all days of one year at one place.

`.corr()` gives the **correlation** between columns, a number from −1 to 1. Near 1, the two series rise and fall together; near 0, they are unrelated.

In [ ]:
wide_mm = weather.pivot(index="date", columns="place", values="rain_mm")
print(wide_mm.head(3))
print(wide_mm.shape)
print(wide_mm.corr().round(3))

In [ ]:
annual_both_mm = weather.pivot_table(index="year", columns="place", values="rain_mm", aggfunc="sum")
print(annual_both_mm.head(3).round(1))
print(annual_both_mm.mean().round(1))

The wide table has 8,766 rows and two columns, Kathmandu and Melamchi. The daily correlation is 0.947, very high: on most days both places are wet or dry together.

Be careful with what this means. The two values are averages over two neighbouring 25–30 km squares of the same model, and averaging makes neighbours alike. Real rain gauges in two valleys can disagree far more on single days, because one storm can soak one valley and miss the next.

The same caution applies to the totals: in this data Kathmandu (2,877.6 mm a year) comes out wetter than Melamchi (2,656.5 mm). Do not quote such model values as facts about the two towns. Before you rely on them, compare them with rain-gauge records for the same places.

### Practice

**Q1 · one line:** Correlate the **monthly** totals of the two places: resample `wide_mm` to months with sums, then use `.corr()`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
wide_mm.resample("ME").sum().corr().round(3)
```

0.994, even higher than for days: monthly totals are driven by the monsoon season, which both places share.
</details>

**Q2 · two lines:** Draw a scatter plot of daily rainfall, Kathmandu on the x-axis and Melamchi on the y-axis, with axis labels in mm.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(wide_mm["Kathmandu"], wide_mm["Melamchi"], s=4, alpha=0.3)
ax.set_xlabel("Kathmandu (mm per day)")
ax.set_ylabel("Melamchi (mm per day)")
plt.show()
```

The dots lie along a rising band: wet days at one place are wet days at the other. `s=4` makes small dots and `alpha=0.3` makes them see-through, so the 8,766 days do not hide each other.
</details>

**Q3 · three lines:** Make a Series `diff_mm`: Melamchi minus Kathmandu, from the columns of `wide_mm`. Find the day with the largest difference, then print both places' rainfall on that day.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
diff_mm = wide_mm["Melamchi"] - wide_mm["Kathmandu"]
biggest_day = diff_mm.idxmax()
print(biggest_day.date(), wide_mm.loc[biggest_day])
```

On 2009-07-01, Melamchi had 91.9 mm and Kathmandu 24.9 mm, a difference of 67.0 mm. Even in smoothed model data, a wet day at one place can be a moderate day at the other.
</details>

## 8. Extremes, and what a 25 km model cannot see

For hazards, extremes matter more than averages. Three tools help:

- `series.nlargest(5)`: the five largest values, with their dates,
- a mask such as `rain_mm > 100`, grouped by year and summed: the number of days above 100 mm in each year (each `True` counts as 1, as in A8),
- `rain_mm.resample("YE").max()`: the wettest day of each year, called the **annual maximum**.

Now the honest part. The facts sheet tells you the Melamchi flood happened in June 2021, in Sindhupalchok. **Predict:** will June 2021 stand out as extreme in this data?

In [ ]:
print(rain_mm.nlargest(5))

heavy_days = (rain_mm > 100).groupby(rain_mm.index.year).sum()
print("days above 100 mm, 2019 to 2023:", list(heavy_days.loc[2019:2023]))
print("in all 24 years:", heavy_days.sum())

annual_max_mm = rain_mm.resample("YE").max()
print("smallest annual maximum:", annual_max_mm.min(), "mm, in", annual_max_mm.idxmin().year)

In [ ]:
june_max_mm = rain_mm.loc["2021-06"].max()
print("wettest day of June 2021:", june_max_mm, "mm")
print("days in 2001–2024 with at least as much rain:", (rain_mm >= june_max_mm).sum())
print("every year has a wetter day:", (annual_max_mm > june_max_mm).all())

colours = np.where(annual_max_mm.index.year == 2021, "tab:red", "tab:blue")
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(annual_max_mm.index.year, annual_max_mm, color=colours)
ax.axhline(june_max_mm, color="black", linestyle="--", label="Wettest day of June 2021")
ax.set_title("Wettest day of each year at Melamchi (ERA5 reanalysis; 2021 in red)")
ax.set_xlabel("Year")
ax.set_ylabel("Rainfall (mm per day)")
ax.legend()
plt.show()

The data are clear: June 2021 does **not** stand out. Its wettest day had 58.8 mm, and 140 days of the record, this one included, had at least as much. Every year, even the driest, has a wetter day. Its largest 3-day total, 100.4 mm, is far below the record's 445.3 mm. 2021 was a wet year overall, but its wettest days came in July and August.

That does not mean the flood had nothing to do with rain. It means **this dataset cannot see it**. Each value is an average over a square about 25–30 km wide. A storm that pours down on a few square kilometres of a high valley is spread out over the whole square and looks moderate. Also, a flood in a valley can be set off by processes far upstream, which daily rainfall at one place does not record. Explaining this flood is work for specialists with field data; it is not something you can read from this table.

The general lesson: **a dataset can only show what it was built to measure.** Before you use any data to study an event, ask how large an area each value stands for, and whether the event is larger or smaller than that.

### Practice

**Q1 · one line:** Print the five wettest days at Kathmandu, using the `Kathmandu` column of `wide_mm`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(wide_mm["Kathmandu"].nlargest(5))
```

Kathmandu's wettest day in this data is 2017-08-12, with 270.1 mm. Its top five days fall in July and August, the core of the monsoon.
</details>

**Q2 · two lines:** Which years had the most days above 100 mm at Melamchi? Print every year whose count in `heavy_days` equals the largest count.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
most = heavy_days.max()
print(most, list(heavy_days[heavy_days == most].index))
```

Three years tie with 4 days each: 2007, 2017 and 2021. `idxmax` would have given only the first of them, 2007; a mask finds all ties.
</details>

**Q3 · three lines:** For each month of 2021, print the wettest day's rainfall at Melamchi. Use `.loc["2021"]` and then `resample("ME").max()`. In which month was the wettest day of 2021?

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
max_2021_mm = rain_mm.loc["2021"].resample("ME").max()
print(max_2021_mm)
print(rain_mm.loc["2021"].idxmax().date())
```

The wettest day of 2021 was 2 July, with 176.3 mm, three times June's 58.8 mm. Even May (90.7 mm) and October (76.1 mm) had wetter days than June. The month of the flood is not the month of the wettest day in this data.
</details>

## 9. When it breaks: selecting by date on a text index

You can forget `parse_dates`. The table looks the same, so nothing warns you until you select by date. The cell below uses `raw`, the table read in step 1 without `parse_dates`. Run it.

In [ ]:
mel_text = raw[raw["place"] == "Melamchi"].set_index("date")
mel_text.loc["2021-06"]

Scroll to the last line: `KeyError: '2021-06'`.

- A `KeyError` means a label was not found (A4, A9).
- The index is **text**, so `.loc` looks for a row label that is exactly the text `"2021-06"`. No row has that label; the labels are full dates such as `"2021-06-15"`.
- Only a **DatetimeIndex** understands that `"2021-06"` means every day of June 2021.

To check, print `type(mel_text.index[0])`: it says `str`. With a DatetimeIndex it says `Timestamp`.

**Try it:** fix the cell by converting the column before setting the index: `raw["date"] = pd.to_datetime(raw["date"])`. Then run it again. Better still, use `parse_dates=["date"]` every time you read a file with dates.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · Kathmandu in numbers

Take the `Kathmandu` column of `wide_mm` as a Series. Compute its annual totals with `resample`, print the mean, the driest and the wettest year with their totals, and draw a bar chart of the anomalies (each year minus the mean), with blue for wet and orange for dry years. Add a title that names the data source and labelled axes.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
ktm_mm = wide_mm["Kathmandu"]
ktm_annual_mm = ktm_mm.resample("YE").sum()
print("mean:", round(ktm_annual_mm.mean(), 1), "mm")
print("driest:", ktm_annual_mm.idxmin().year, round(ktm_annual_mm.min(), 1))
print("wettest:", ktm_annual_mm.idxmax().year, round(ktm_annual_mm.max(), 1))

ktm_anomaly_mm = ktm_annual_mm - ktm_annual_mm.mean()
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(ktm_anomaly_mm.index.year, ktm_anomaly_mm, color=np.where(ktm_anomaly_mm > 0, "tab:blue", "tab:orange"))
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Annual rainfall at Kathmandu: difference from the 2001–2024 mean (ERA5)")
ax.set_xlabel("Year")
ax.set_ylabel("Anomaly (mm)")
plt.show()
```

The mean is 2,877.6 mm; the driest year is 2015 (2,000.1 mm) and the wettest 2021 (4,159.6 mm). Both places have the same driest and wettest years, as you would expect from neighbouring squares of the same model.
</details>

### Review 2 · How much of each year falls in the monsoon?

Using `monthly_mm`, keep the months from June to September (month numbers 6 to 9, two conditions joined with `&`, A9). Add them up per year with `resample("YE").sum()`, divide by `annual_mm` and multiply by 100. Print the lowest and the highest share with their years, then draw the shares as a line with markers.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
month_no = monthly_mm.index.month
monsoon_mm = monthly_mm[(month_no >= 6) & (month_no <= 9)].resample("YE").sum()
monsoon_pct = monsoon_mm / annual_mm * 100
print("lowest:", monsoon_pct.idxmin().year, round(monsoon_pct.min(), 1), "%")
print("highest:", monsoon_pct.idxmax().year, round(monsoon_pct.max(), 1), "%")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(monsoon_pct.index.year, monsoon_pct, marker="o")
ax.set_ylim(0, 100)
ax.set_title("Share of each year's rainfall from June to September, Melamchi (ERA5)")
ax.set_xlabel("Year")
ax.set_ylabel("Monsoon share (%)")
plt.show()
```

The share ranges from 69.5% (2004) to 86.0% (2010). Both Series are labelled with year-end dates, so pandas lines them up by date when it divides. The y-axis from 0 to 100 shows the shares as parts of a whole (A10).
</details>

### Review 3 · The wettest week of every year

For each year, find the largest 7-day total at Melamchi: resample `rain_7d_mm` to years with `.max()`. Print the three years with the wettest weeks with `nlargest(3)`, and the 2021 value. Then draw a bar chart with 2021 in red.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
week_max_mm = rain_7d_mm.resample("YE").max()
print(week_max_mm.nlargest(3).round(1))
print("2021:", round(week_max_mm.loc["2021"].iloc[0], 1), "mm")

colours = np.where(week_max_mm.index.year == 2021, "tab:red", "tab:blue")
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(week_max_mm.index.year, week_max_mm, color=colours)
ax.set_title("Largest 7-day rainfall of each year at Melamchi (ERA5; 2021 in red)")
ax.set_xlabel("Year")
ax.set_ylabel("7-day rainfall (mm)")
plt.show()
```

The wettest weeks are in 2007 (618.4 mm), 2019 (610.6 mm) and 2017 (606.1 mm); 2021's wettest week had 444.8 mm. `.loc["2021"]` on a yearly Series still returns a Series with one value, so `.iloc[0]` takes that value out. A rolling total followed by a yearly maximum is a common way to describe antecedent rainfall in hazard studies.
</details>

### Review 4 · A monthly table for a report

Make a wide table of **monthly** rainfall totals for both places: resample `wide_mm` with `"MS"` (labels on the first day of each month) and `.sum()`, and round to 1 decimal place. Save it as `rain_monthly_two_places.csv` in `OUT`, read it back with `parse_dates=["date"]` and `index_col="date"`, print its shape and the rows for June to September 2021.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
monthly_both_mm = wide_mm.resample("MS").sum().round(1)
monthly_both_mm.to_csv(OUT / "rain_monthly_two_places.csv")

check = pd.read_csv(OUT / "rain_monthly_two_places.csv",
                    parse_dates=["date"], index_col="date")
print(check.shape)
print(type(check.index[0]))
print(check.loc["2021-06":"2021-09"])
```

The file has 288 rows, one per month, and two columns. Here the dates are real data, so the index is saved with the table (A9). `index_col="date"` makes them the index again and `parse_dates` makes it a DatetimeIndex, so selecting a range of months works on the table you read back. June 2021 shows 519.8 mm at Melamchi, as in step 3.
</details>

## Recap

- Dates in a CSV file are text until you convert them: use `parse_dates=["date"]` in `read_csv` or `pd.to_datetime`, with `format=` for day-first dates. Subtracting dates gives a `Timedelta`.
- `.dt.year`, `.dt.month`, `.dt.day_of_year` and `.dt.day_name()` take dates apart; with `set_index("date")`, `.loc["2021-06"]` selects a month and `.idxmax()` finds the date of the largest value.
- `resample("ME")` and `resample("YE")` turn days into months and years in time order; `groupby(index.month)` builds the average year. Sum amounts like rain; average states like temperature.
- `rolling(3).sum()` gives antecedent rainfall over a moving window; `cumsum()` gives running totals; `pivot` and `pivot_table` turn a long table into a wide one for comparing places.
- Reanalysis values are averages over squares 25–30 km wide: they smooth out local extremes. The June 2021 Melamchi flood is not visible as an extreme in this data, and a dataset can only show what it was built to measure.

### Words you learned

| Word | Meaning |
| --- | --- |
| time series | Values recorded one after another in time, such as daily rainfall |
| reanalysis | A weather model corrected with observations, giving estimates for every place and day; ERA5 is one |
| ERA5 | The reanalysis of the Copernicus Climate Change Service used in this lesson; its cells are about 25–30 km wide |
| Timestamp | pandas' type for one date (and time) |
| datetime column | A column of Timestamps, which pandas can sort, subtract and take apart |
| `parse_dates` | A `read_csv` option that converts the named columns to dates while reading |
| `pd.to_datetime` | Converts text to dates; `format=` describes the pattern, such as `"%d/%m/%Y"` |
| ISO 8601 | The international way of writing dates: year-month-day, such as 2021-06-15 |
| Timedelta | A length of time, the result of subtracting two Timestamps |
| accessor | A group of tools behind a dot name, such as `.dt` for dates |
| leap year | A year with 366 days, including 29 February |
| DatetimeIndex | Row labels that are dates, so `.loc` can select by day, month or year |
| `idxmax` / `idxmin` | Give the label (here, the date) of the largest / smallest value |
| `resample` | Groups a time series by period (`"ME"`, `"MS"`, `"YE"`) and summarises each period |
| anomaly | The difference between a value and the long-term mean |
| climatology | The average year: the mean of each month over many years |
| antecedent rainfall | Rain of the days before an event, which wets the ground |
| rainfall threshold | An amount of rain over some days above which landslides become likely in a region, found from local records |
| rolling window | A window of fixed length that moves along the series, such as the last 3 days |
| `NaN` | "Not a number": pandas' mark for a missing value |
| running total (`cumsum`) | The sum of all values so far |
| long / wide table | One row per place and day / one row per day and one column per place |
| `pivot` / `pivot_table` | Turn a long table into a wide one; `pivot_table` also summarises with `aggfunc` |
| correlation | A number from −1 to 1 that says how closely two series rise and fall together |
| annual maximum | The largest value of each year, such as the wettest day |

## Next

**A12 · Scripts, modules and good habits:** `while` loops, tuples and sets, your own module of functions, scripts, finding help on your own, and the habits that make your results trustworthy.

<!-- pygile-nav -->
**PyGILE Course** · lesson 11 of 39 · [Course home](../README.md)  
← Previous: [A10 · Plotting with Matplotlib: charts of rain and mountains](A10_Plotting_with_Matplotlib.ipynb) · Next: [A12 · Scripts, modules and good habits: tools for real projects](A12_Scripts_modules_and_good_habits.ipynb) →